[目录](./table_of_contents.ipynb)

# 设计非线性卡尔曼滤波器

In [ ]:
%matplotlib inline

In [ ]:
#格式化书本
import book_format
book_format.set_style()

## 介绍

**作者注: 我最初计划有一个设计非线性章节来比较各种方法。这可能会发生，也可能不会发生，但现在这个章节没有有用的内容，建议不要阅读。**

我们看到卡尔曼滤波器可以合理地跟踪球的运动。然而，正如已经解释过的那样，这是一个愚蠢的例子；我们可以使用任意精度在真空中预测轨迹；在这个例子中使用卡尔曼滤波器是一种不必要的复杂化。

### 带有空气阻力的卡尔曼滤波器

我将不采用步骤1、步骤2、类型的方法，而是采用更自然的方式进行处理，这是在非玩具工程问题中使用的方式。我们已经开发了一个卡尔曼滤波器，可以在真空中非常好地跟踪球，但是它没有将空气阻力的影响纳入模型中。我们知道过程模型是用 $\textbf{F}$ 实现的，因此我们将立即转向它。

概念上，$\textbf{F}$ 计算的计算是

$$x' = Fx$$

没有空气阻力，我们有

$$
\mathbf{F} = \begin{bmatrix}
1 & \Delta t & 0 & 0 & 0 \\
0 & 1 & 0 & 0 & 0 \\
0 & 0 & 1 & \Delta t & \frac{1}{2}{\Delta t}^2 \\
0 & 0 & 0 & 1 & \Delta t \\
0 & 0 & 0 & 0 & 1
\end{bmatrix}
$$

对应于以下方程

$$ 
\begin{aligned}
x &= x + v_x \Delta t \\
v_x &= v_x \\
\\
y &= y + v_y \Delta t + \frac{a_y}{2} {\Delta t}^2 \\
v_y &= v_y + a_y \Delta t \\
a_y &= a_y
\end{aligned}
$$

从上面的部分，我们知道我们的新欧拉方程必须是：

$$ 
\begin{aligned}
x &= x + v_x \Delta t \\
v_x &= v_x \\
\\
y &= y + v_y \Delta t + \frac{a_y}{2} {\Delta t}^2 \\
v_y &= v_y + a_y \Delta t \\
a_y &= a_y
\end{aligned}
$$

## 现实中的二维位置传感器

上一个例子中的位置传感器并不是非常现实。通常情况下，没有提供 (x,y) 坐标的“原始”传感器。我们有 GPS，但 GPS 已经使用卡尔曼滤波器创建了一个滤波输出；除非我们加入其他传感器来提供额外的信息，否则我们不应该通过另一个卡尔曼滤波器来改善信号。我们将在以后解决这个问题。

考虑以下设置。在一个开阔的田野里，我们放置了两个发射器在已知位置，每个发射器都发射我们可以检测到的信号。我们处理信号并确定我们离信号有多远，有一些噪声。首先，让我们看一下其视觉描述。

In [ ]:
import matplotlib.pyplot as plt


circle1=plt.Circle((-4, 0), 5, color='#004080', 
                   fill=False, linewidth=20, alpha=.7)
circle2=plt.Circle((4, 0), 5, color='#E24A33', 
                   fill=False, linewidth=5, alpha=.7)

fig = plt.gcf()
ax = fig.gca()

plt.axis('equal')
plt.xlim((-10, 10))
plt.ylim((-10, 10))

plt.plot ([-4, 0], [0, 3], c='#004080')
plt.plot ([4, 0], [0, 3], c='#E24A33')
plt.text(-4, -.5, "A", fontsize=16, horizontalalignment='center')
plt.text(4, -.5, "B", fontsize=16, horizontalalignment='center')

ax.add_artist(circle1)
ax.add_artist(circle2)
plt.show()

In [ ]:

这里我试图展示发射器A（红色）位于(-4,0)，第二个发射器B（蓝色）位于(4,0)。红色和蓝色圆圈显示了从发射器到机器人的范围，其宽度说明了每个发射器的$1\sigma$角度误差的影响。这里我给蓝色发射器比红色的误差更大。机器人最可能的位置是两个圆圈相交的地方，我用红色和蓝色的线描绘了出来。你会反对说我们有两个交点，而不是一个，但我们将在设计测量函数时看看如何处理它。

这是一个非常常见的传感器设置。飞机仍然使用这个系统进行导航，称为DME（距离测量设备）。今天，GPS是一种更常见的导航系统，但我曾经在一架飞机上工作过，在那里我们将这样的传感器与GPS、INS、高度计等集成到我们的滤波器中。我们将在以后处理所谓的*多传感器融合*；现在我们只处理这个简单的配置。

第一步是设计我们的状态变量。我们将假设机器人以恒定速度直线行驶。这对于长时间来说不太可能是真的，但对于短时间来说是可以接受的。这与上一个问题没有区别——我们将要跟踪机器人的位置和速度。因此，

$$\mathbf{x} = 
\begin{bmatrix}x\\v_x\\y\\v_y\end{bmatrix}$$

下一步是设计状态转移函数。这也与上一个问题相同，因此不再赘述，

$$
\mathbf{x}' = \begin{bmatrix}1& \Delta t& 0& 0\\0& 1& 0& 0\\0& 0& 1& \Delta t\\ 0& 0& 0& 1\end{bmatrix}\mathbf{x}$$

下一步是设计控制输入。我们没有输入，所以将 ${\mathbf{B}}$ 设为0。

下一步是设计测量函数 $\mathbf{z} = \mathbf{Hx}$。我们可以使用勾股定理来建模测量过程。

$$
z_a = \sqrt{(x-x_A)^2 + (y-y_A)^2} + v_a\\[1em]
z_b = \sqrt{(x-x_B])^2 + (y-y_B)^2} + v_b
$$

其中 $v_a$ 和 $v_b$ 是白noise。

我们立刻遇到了一个问题。KalmanFilter是为线性方程设计的，而这显然是非线性的。在接下来的章节中，我们将探讨处理非线性问题的几种方法，但现在我们将做一些更简单的事情。如果我们知道机器人的近似位置，那么我们可以在线性化该点周围的方程。我现在可以为这种技术开发广义数学，但是我想呈现一个已解决的例子来解释这个技术。

我们将计算 $\mathbf{H}$ 的偏导数，而不是直接计算它。 偏导数是指 $\mathbf{H}$ 相对于机器人位置 $\mathbf{x}$ 的变化量。 具体计算方法如下：

$$\frac{\partial \mathbf{h}}{\partial \mathbf{x}} = 
\begin{bmatrix}
\frac{\partial h_1}{\partial x_1} & \frac{\partial h_1}{\partial x_2} &\dots \\
\frac{\partial h_2}{\partial x_1} & \frac{\partial h_2}{\partial x_2} &\dots \\
\vdots & \vdots
\end{bmatrix}
$$

我们先找出第一个偏导数

$$\frac{\partial }{\partial x} \sqrt{(x-x_A)^2 + (y-y_A)^2}
$$

我们计算得到

$$
\begin{aligned}
\frac{\partial h_1}{\partial x} &= ((x-x_A)^2 + (y-y_A)^2))^\frac{1}{2} \\
&= \frac{1}{2}\times 2(x-x_a)\times ((x-x_A)^2 + (y-y_A)^2))^{-\frac{1}{2}} \\
&= \frac{x_r - x_A}{\sqrt{(x_r-x_A)^2 + (y_r-y_A)^2}} 
\end{aligned}
$$

我们继续计算另外两个距离方程的偏导数，分别对$x$，$y$，$dx$和$dy$ 求偏导数，得到:

$$\frac{\partial\mathbf{h}}{\partial\mathbf{x}}=
\begin{bmatrix}
\frac{x_r - x_A}{\sqrt{(x_r-x_A)^2 + (y_r-y_A)^2}} & 0 & 
\frac{y_r - y_A}{\sqrt{(x_r-x_A)^2 + (y_r-y_A)^2}} & 0 \\
\frac{x_r - x_B}{\sqrt{(x_r-x_B)^2 + (y_r-y_B)^2}} & 0 &
\frac{y_r - y_B}{\sqrt{(x_r-x_B)^2 + (y_r-y_B)^2}} & 0 \\
\end{bmatrix}
$$

这真是太痛苦了，这些是非常简单的方程。对于更复杂的系统，计算雅可比矩阵可能会极其困难甚至不可能。然而，有一个简单的方法可以让Python为您完成这项工作，即使用SymPy模块[1]。SymPy是一个用于符号数学的Python库。它的全部功能超出了本书的范围，但它可以执行代数、积分和微分方程、寻找微分方程的解等等。我们将使用它来计算我们的雅可比矩阵！

首先是一个简单的例子。我们将导入 SymPy，初始化其漂亮的打印功能（它将使用 LaTeX 打印方程式）。然后，我们将声明一个符号供 NumPy 使用。

```python
import sympy
from sympy import init_printing
init_printing(use_latex='mathjax')

phi, x = sympy.symbols('\phi, x')
phi

注意我们如何为符号“phi”使用 LaTeX 表达式。这不是必要的，但如果这样做，它将在输出时呈现为 LaTeX。现在让我们进行一些数学计算。$\sqrt{\phi}$的导数是什么？

In [ ]:
sympy.diff('sqrt(phi)')

我们可以因式分解方程式。

In [ ]:
sympy.factor('phi**3 -phi**2 + phi - 1')

SymPy具有出色的功能列表，尽管我喜欢使用其功能，但我们无法在此处涵盖所有内容。相反，让我们计算我们的Jacobian。

In [ ]:
from sympy import symbols, Matrix
phi = symbols('\phi')
phi

x, y, xa, xb, ya, yb, dx, dy = symbols('x y x_a x_b y_a y_b dx dy')

H = Matrix([[sympy.sqrt((x-xa)**2 + (y-ya)**2)], 
            [sympy.sqrt((x-xb)**2 + (y-yb)**2)]])

state = Matrix([x, dx, y, dy])
H.jacobian(state)

In [ ]:
简而言之，(0,0)条目包含机器人和发射机 A 的 x 坐标之间的差异，除以机器人和 A 之间的距离。(2,0)包含相同的内容，只是机器人和发射机的 y 坐标不同。底部行中包含相同的计算，除了发射机 B。0 条目包括状态变量的速度分量；自然地，距离不提供我们速度。

这个矩阵中的值随着机器人位置的变化而变化，因此这不再是一个常量；我们将不得不为滤波器的每个时间步骤重新计算它。

如果您查看此内容，您可能会意识到这只是 x/dist 和 y/dist 的计算，因此我们可以将其转换为三角形式，而不会失去一般性：

$$\frac{\partial\mathbf{h}}{\partial\mathbf{x}}=
\begin{bmatrix}
-\cos{\theta_A} & 0 & -\sin{\theta_A} & 0 \\
-\cos{\theta_B} & 0 & -\sin{\theta_B} & 0
\end{bmatrix}
$$

然而，这引发了一个巨大的问题。我们不再计算 $\mathbf{H}$，而是 $\Delta\mathbf{H}$，即 $\mathbf{H}$ 的变化量。如果我们在不改变其他设计的情况下将其传递到我们的KalmanFilter中，输出将是无意义的。请记住，例如，我们将 $\mathbf{Hx}$ 相乘以生成从给定 $\mathbf{x}$ 估计结果产生的测量值。但现在，由于 $\mathbf{H}$ 在我们的位置周围进行了线性化，因此它包含测量函数的*变化量*。

因此，我们被迫使用 $\mathbf{x}$ 的*变化量*作为我们的状态变量。因此，我们必须返回并重新设计我们的状态变量。

请注意，设计KalmanFilter时，这是一个完全正常的情况。教科书将这样的例子呈现为*事实*，好像很显然状态变量需要是速度而不是位置。也许当你做足够多的这些问题时，它会变得显而易见，但此时你为什么还要阅读教科书呢？我发现自己会多次阅读演示文稿，试图弄清楚为什么他们做出了某个选择，最终才意识到这是因为下一页的某些后果。我的演示文稿更长，但它反映了当你设计滤波器时实际发生的情况。您做出看起来合理的设计选择，并且随着您向前移动，您会发现需要重新制定早期步骤的属性。因此，我将在某种程度上放弃我的**第1步**，**第2步**等方法，因为许多实际问题并不那么简单。

如果我们的状态变量包含机器人的速度而不是位置，那么我们如何跟踪机器人的位置呢？我们做不到。在这种情况下线性化的KalmanFilter使用所谓的*名义轨迹* - 即您假设一个位置和方向，然后应用速度和加速度的变化来计算轨迹的变化。难道还有其他方式吗？回想一下显示两个测距圆之间交点的图形 - 有两个交点区域。想象一下如果两个发射器非常接近会是什么样子 - 交点将是两个非常长的新月形状。这个KalmanFilter，按设计，没有办法仅通过与发射器的距离测量来知道您的真实位置。也许您的头脑已经开始想办法解决这个问题了。如果是这样，请继续保持关注，因为后面的章节将为您提供这些技术。一次性呈现全部解决方案并不可取。

在我看来，这只会增加更多的混乱而不是洞见。

因此，让我们重新设计我们的*状态转移函数*。我们假设速度恒定，没有加速度，给出状态方程式为

$$
\dot{x}' = \dot{x} \\
\ddot{x}' = 0 \\
\dot{y}' = \dot{y} \\
\dot{y}' = 0$$

这给了我们*状态转移函数*

$$
\mathbf{F} = \begin{bmatrix}0 &1 & 0& 0\\0& 0& 0& 0\\0& 0& 0& 1\\ 0& 0& 0& 0\end{bmatrix}$$

最后一个复杂性来自于我们传递的测量。$\mathbf{Hx}$ 现在计算的是测量从我们的标称位置的变化，所以我们传递的测量需要不是到 A 和 B 的距离，而是从我们的测量距离到我们的标称位置的*变化*。

这里有很多东西需要理解，所以让我们逐个代码逐步解决。首先，我们将定义一个函数来计算每个时间步长的 $\frac{\partial\mathbf{h}}{\partial\mathbf{x}}$。

```python
from math import sin, cos, atan2

In [ ]:
def H_of(pos, pos_A, pos_B):
    """给出我们在2D中的物体在'pos'处的位置，以及两个发射器A和B在位置'pos_A'和'pos_B'，返回H的偏导数"""
    
    theta_a = atan2(pos_a[1] - pos[1], pos_a[0] - pos[0])
    theta_b = atan2(pos_b[1] - pos[1], pos_b[0] - pos[0])

    return np.array([[0, -cos(theta_a), 0, -sin(theta_a)],
                     [0, -cos(theta_b), 0, -sin(theta_b)]])

现在我们需要创建模拟传感器。

In [ ]:
from numpy.random import randn

In [ ]:
class DMESensor(object):
    def __init__(self, pos_a, pos_b, noise_factor=1.0):
        self.A = pos_a
        self.B = pos_b
        self.noise_factor = noise_factor
        
    def range_of(self, pos):
        """ 返回一个元组，其中包含到A和B的带有noise的距离data，给定一个位置'pos'"""
        
        ra = math.sqrt((self.A[0] - pos[0])**2 + (self.A[1] - pos[1])**2)
        rb = math.sqrt((self.B[0] - pos[0])**2 + (self.B[1] - pos[1])**2)
        
        return (ra + randn()*self.noise_factor, 
                rb + randn()*self.noise_factor)

最后，我们准备好了Kalman滤波器代码。我将在x=-100和100处放置发射器，y值都为-20。这样我就有足够的空间从两个发射器的三角测量中得到好的结果，当机器人移动时，我将在(0,0)处开始并每个时间步长移动(1,1)。

In [ ]:
import kf_book.book_plots as bp
from filterpy.kalman import KalmanFilter
import math
import numpy as np

pos_a = (100, -20)
pos_b = (-100, -20)

f1 = 卡尔曼滤波器(dim_x=4, dim_z=2)

f1.F = np.array ([[0, 1, 0, 0],
                  [0, 0, 0, 0],
                  [0, 0, 0, 1],
                  [0, 0, 0, 0]], dtype=float)

f1.R *= 1.
f1.Q *= .1

f1.x = np.array([[1, 0, 1, 0]], dtype=float).T
f1.P = np.eye(4) * 5.

# 初始化存储和运行所需的变量
count = 30
xs, ys = [], []
pxs, pys = [], []

# 创建模拟传感器
d = DMESensor(pos_a, pos_b, noise_factor=3.)

# pos 将存储我们的名义位置，因为滤波器不维护位置。
pos = [0, 0]

for i in range(count):
    # 每步移动（1,1），所以只需使用 i
    pos = [i, i]

In [ ]:
# 计算名义轨迹和测量距离之间的差异
ra,rb = d.range_of(pos)
rx,ry = d.range_of((pos[0] + f1.x[0, 0], pos[1] + f1.x[2, 0]))
z = np.array([[ra - rx], [rb - ry]])

# 计算此时间步长的线性化 H
f1.H = H_of (pos, pos_a, pos_b)

# 存储数据以便之后绘图
    xs.append(f1.x[0, 0]+i)
    ys.append(f1.x[2, 0]+i)
    pxs.append(pos[0])
    pys.append(pos[1])

In [ ]:
# 执行卡尔曼滤波步骤
f1.predict()
f1.update(z)

bp.plot_filter(xs, ys)
bp.plot_track(pxs, pys)
plt.legend(loc=2)
plt.show()

In [ ]:

## 线性化卡尔曼滤波器

现在我们已经看到了线性化KalmanFilter的一个例子，我们可以更好地理解数学。 

我们首先假设一些函数 $\mathbf f$

## 例子：下落的球

**作者注：现在先忽略这一节。**

在**设计KalmanFilter**章节中，我首先考虑了在真空中跟踪一个球，然后是在大气中。KalmanFilter在真空中表现非常好，但在大气中偏离了球的轨迹。让我们看一下输出；为了避免在这一章中插入来自那一章的代码，我把它们都放在了文件 `ekf_internal.py` 中。

```python
import kf_book.ekf_internal as ekf
ekf.plot_ball()

我们可以通过将$Q$设为较大的值来人为迫使Kalman滤波器跟踪球的运动。这将导致滤波器不信任其预测，并缩放卡尔曼增益$K$以强烈偏向测量值。然而，这并不是一种有效的方法。如果Kalman滤波器能够正确地预测过程，我们不应该通过告诉它存在不存在的过程误差来“欺骗”滤波器。我们可能可以在某些问题和某些条件下做到这一点，但一般来说，Kalman滤波器的性能会较差。

回想一下**设计Kalman滤波器**一章中加速度的计算公式：

$$a_x = (0.0039 + \frac{0.0058}{1+\exp{[(v-35)/5]}})*v*v_x \\
a_y = (0.0039 + \frac{0.0058}{1+\exp{[(v-35)/5]}})*v*v_y- g
$$

这些方程在我们发展这个主题时将非常棘手，因此现在我将退回到使用这个简化的加速度方程的更简单的一维问题，该方程不考虑阻力系数的非线性性：

$$\begin{aligned}
\ddot{y} &= \frac{0.0034ge^{-y/20000}\dot{y}^2}{2\beta} - g \\
\ddot{x} &= \frac{0.0034ge^{-x/20000}\dot{x}^2}{2\beta}
\end{aligned}$$

这里，$\beta$ 是弹道系数，高数值表示低阻力。

这仍然是非线性的，因此我们需要在当前状态点上将此方程线性化。如果我们的状态是位置和速度，我们需要一个关于 $\mathbf{x}$ 中任意小变化的方程，如下所示：

$$ \begin{bmatrix}\Delta \dot{x} \\ \Delta \ddot{x} \\ \Delta \dot{y} \\ \Delta \ddot{y}\end{bmatrix} = 
\large\begin{bmatrix}
\frac{\partial \dot{x}}{\partial x} & 
\frac{\partial \dot{x}}{\partial \dot{x}} & 
\frac{\partial \dot{x}}{\partial y} & 
\frac{\partial \dot{x}}{\partial \dot{y}} \\ 
\frac{\partial \ddot{x}}{\partial x} & 
\frac{\partial \ddot{x}}{\partial \dot{x}}& 
\frac{\partial \ddot{x}}{\partial y}& 
\frac{\partial \dot{x}}{\partial \dot{y}}\\
\frac{\partial \dot{y}}{\partial x} & 
\frac{\partial \dot{y}}{\partial \dot{x}} & 
\frac{\partial \dot{y}}{\partial y} & 
\frac{\partial \dot{y}}{\partial \dot{y}} \\ 
\frac{\partial \ddot{y}}{\partial x} & 
\frac{\partial \ddot{y}}{\partial \dot{x}}& 
\frac{\partial \ddot{y}}{\partial y}& 
\frac{\partial \dot{y}}{\partial \dot{y}}
\end{bmatrix}\normalsize
\begin{bmatrix}\Delta x \\ \Delta \dot{x} \\ \Delta \dot{y} \\ \Delta \ddot{y}\end{bmatrix}$$

方程中不同时包含 $x$ 和 $y$，因此任何同时包含二者的偏导数必须等于零。我们还知道 $\large\frac{\partial \dot{x}}{\partial x}\normalsize = 0$，$\large\frac{\partial \dot{x}}{\partial \dot{x}}\normalsize = 1$，因此我们的矩阵为

$$\mathbf{F} = \begin{bmatrix}0&1&0&0 \\
\frac{0.0034e^{-x/22000}\dot{x}^2g}{44000\beta}&0&0&0
\end{bmatrix}$$



$$\begin{aligned}\ddot{x} &= -\frac{1}{2}C_d\rho A \dot{x}\\
\ddot{y} &= -\frac{1}{2}C_d\rho A \dot{y}-g\end{aligned}$$

In [ ]:
from sympy.abc import *
from sympy import *

init_printing(pretty_print=True, use_latex='mathjax')

x1 = (0.0034*g*exp(-x/22000)*((x)**2))/(2*b) - g

x2 = (a*g*exp(-x/c)*(Derivative(x)**2))/(2*b) - g

#pprint(x1)
#pprint(Derivative(x)*Derivative(x,n=2))
#pprint(diff(x2, x))

**孤儿文本

这种方法存在许多问题。首先，线性化并不能产生精确的答案。更重要的是，我们线性化的不是实际路径，而是我们滤波器对路径的估计。我们线性化估计值是因为它在统计上可能是正确的，但当然并不是必须的。因此，如果滤波器的输出不好，这将导致我们线性化一个不正确的估计，这几乎肯定会导致更糟糕的估计。在这些情况下，滤波器将快速发散。这就是卡尔曼滤波器的“黑魔法”之处。我们正在尝试线性化一个估计，而并不能保证滤波器是稳定的。大量有关卡尔曼滤波器的文献都致力于解决这个问题。另一个问题是，我们需要使用分析方法来线性化系统。对于某些问题，可能难以或不可能找到分析解。在其他情况下，我们可能能够找到线性化，但计算可能会很困难。

非常昂贵。**

## 参考资料

[1] http://sympy.org